<a href="https://colab.research.google.com/github/aggelikntaliani-lgtm/scRNA-seq-Rstudio/blob/main/scRNAseq_no_output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scanpy

In [ ]:
import scanpy as sc

In [ ]:
!wget https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5226nnn/GSM5226574/suppl/GSM5226574_C51ctr_raw_counts.csv.gz
adata = sc.read_csv('GSM5226574_C51ctr_raw_counts.csv.gz').T
adata

In [ ]:
adata.X.shape

In [ ]:
!pip install scvi-tools

In [ ]:
import scvi

In [ ]:
adata

In [ ]:
sc.pp.filter_genes(adata, min_cells = 10)

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes = 2000, subset = True, flavor = 'seurat_v3')

In [ ]:
scvi.model.SCVI.setup_anndata(adata)
vae = scvi.model.SCVI(adata)
vae.train()

In [ ]:
solo = scvi.external.SOLO.from_scvi_model(vae)
solo.train()

In [ ]:
df = solo.predict()
df['prediction'] = solo.predict(soft = False)
df

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df.groupby('prediction').count()

In [ ]:
df['dif'] = df.doublet - df.singlet
df

In [ ]:
import seaborn as sns

In [ ]:
sns.displot(df[df.prediction == 'doublet'], x = 'dif')

In [ ]:
doublets = df[(df.prediction == 'doublet') & (df.dif > 0.4)]
doublets

In [ ]:
adata = sc.read_csv('GSM5226574_C51ctr_raw_counts.csv.gz').T

In [ ]:
adata.obs['doublet'] = adata.obs.index.isin(doublets.index)

In [ ]:
adata.obs

In [ ]:
adata = adata[~adata.obs.doublet]

In [ ]:
adata

In [ ]:
adata.var['mt'] = adata.var.index.str.startswith('MT-')

In [ ]:
adata.var

In [ ]:
import pandas as pd

In [ ]:
ribo_url = "http://software.broadinstitute.org/gsea/msigdb/download_geneset.jsp?geneSetName=KEGG_RIBOSOME&fileType=txt"

In [ ]:
ribo_genes = pd.read_table(ribo_url, skiprows=2, header = None)
ribo_genes

In [ ]:
adata.var['ribo'] = adata.var_names.isin(ribo_genes[0].values)

In [ ]:
adata.var

In [ ]:
adata.obs

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'ribo'], percent_top= None, log1p= False, inplace=True)

In [ ]:
adata.var.sort_values('n_cells_by_counts')

In [ ]:
sc.pp.filter_genes(adata, min_cells = 3)

In [ ]:
adata.obs.sort_values('total_counts')

In [ ]:
#sc.pp.filter_cells(adata, min_genes = 200)

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo'],
             jitter = 0.4, multi_panel = True)

In [ ]:
import numpy as np

In [ ]:
upper_lim = np.quantile(adata.obs.n_genes_by_counts.values, .98)
#Upper_lim = 3000

In [ ]:
upper_lim

In [ ]:
adata = adata[adata.obs.n_genes_by_counts < upper_lim]

In [ ]:
adata.obs

In [ ]:
adata = adata[adata.obs.pct_counts_mt < 20]

In [ ]:
adata = adata[adata.obs.pct_counts_ribo < 2]

In [ ]:
adata

In [ ]:
#normalization

In [ ]:
adata.X.sum(axis=1)

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4) #normalize every cell to 10000 UMI

In [ ]:
adata.X.sum(axis=1)

In [ ]:
sc.pp.log1p(adata)  #change to log counts

In [ ]:
adata.X.sum(axis=1)

In [ ]:
adata.raw = adata

In [ ]:
#clustering

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes= 2000)

In [ ]:
adata.var

In [ ]:
sc.pl.highly_variable_genes(adata)

In [ ]:
adata = adata [:, adata.var.highly_variable]

In [ ]:
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt', 'pct_counts_ribo'])

In [ ]:
sc.pp.scale(adata, max_value=10)

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)

In [ ]:
sc.pp.neighbors(adata, n_pcs=30)

In [ ]:
adata.obsp['connectivities'].toarray()

In [ ]:
adata.obsp['distances'].toarray()

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata)

In [ ]:
!pip install leidenalg

In [ ]:
sc.tl.leiden(adata, resolution=0.5)

In [ ]:
adata.obs

In [ ]:
sc.pl.umap(adata, color=['leiden'])